In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

In [0]:
silver_df = spark.table(silver_table)

gold_df = (
    silver_df
    .groupBy(F.hour(F.col('dateTime')).alias('hour'))
    .agg(
        F.count('*').alias('transaction_count'),
        F.countDistinct('customerID').alias('customer_count'),
        F.sum('quantity').alias('units_sold'),
        F.sum('totalPrice').alias('total_revenue'),
        F.avg('totalPrice').alias('average_transaction_value')
    )
)

gold_df.write.mode('overwrite').format('delta').saveAsTable(gold_table)

In [0]:
assert gold_df.filter(F.col('dateTime').isNull()).count() == 0, "Null values in dateTime"